# Лабораторная работа №1. Логическая классификация.

## Задание

- выбрать датасет для классификации, например на kaggle;
    - датасет должен содержать пропуски;
    - датасет должен содержать категориальные и количественные признаки;
- реализовать алгоритм построения дерева ID3 с критерием Джини;
- реализовать обработку пропущенных значений через оценку вероятности;
- обучить дерево на выбранном датасете;
- оценить качество классификации;
- реализовать алгоритм редукции дерева;
- сравнить качество классификации и регрессии до и после редукции дерева;
- сравнить с эталонной реализацией бинарного решающего дерева;
    - сравнить качество работы;
- подготовить небольшой отчет о проделанной работе.

## Выполнение ЛР

### Датасет

В качестве датасета был выбран [Titanic Survival Prediction Dataset](https://www.kaggle.com/datasets/yasserh/titanic-dataset)

In [1]:
import kagglehub
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path(kagglehub.dataset_download("yasserh/titanic-dataset")) / "Titanic-Dataset.csv")
df.shape

/home/yomma/itmo/spring-26/students/mavlyukeev-av/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(891, 12)

In [2]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
from sklearn.model_selection import train_test_split

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]
X = df[features].copy()
y = df['Survived'].astype(int).values


In [4]:
X['Sex'] = X['Sex'].map({'female': 0, 'male': 1})
X['Embarked'] = X['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,1,22.0,1,0,7.2500,0.0
1,1,0,38.0,1,0,71.2833,1.0
2,3,0,26.0,0,0,7.9250,0.0
3,1,0,35.0,1,0,53.1000,0.0
4,3,1,35.0,0,0,8.0500,0.0


In [5]:
categorical_idx = [0, 1, 6]
X_arr = X.values

In [6]:
# Train/Validation/Test - 60/20/20

X_train_val, X_test, y_train_val, y_test = train_test_split(X_arr, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.25, random_state=42)

In [7]:
from source.decision_tree import ID3GiniTree

print("Обучение исходного дерева решений (ID3 + Gini)...")
tree = ID3GiniTree(categorical_features=categorical_idx, min_samples_split=2)
tree.fit(X_train, y_train)

Обучение исходного дерева решений (ID3 + Gini)...


In [8]:
y_train_pred_before = tree.predict(X_train)
y_test_pred_before = tree.predict(X_test)

In [9]:
from sklearn.metrics import accuracy_score

acc_train_before = accuracy_score(y_train, y_train_pred_before)
acc_test_before = accuracy_score(y_test, y_test_pred_before)

In [10]:
from source.decision_tree import get_tree_depth, get_leaf_count

depth_before = get_tree_depth(tree.root)
leaves_before = get_leaf_count(tree.root)
acc_test_before = accuracy_score(y_test, tree.predict(X_test))

In [11]:
print("Запуск алгоритма редукции дерева на валидационной выборке...")
tree.prune(X_val, y_val)

Запуск алгоритма редукции дерева на валидационной выборке...


In [12]:
y_train_pred_after = tree.predict(X_train)
y_test_pred_after = tree.predict(X_test)

In [13]:
acc_train_after = accuracy_score(y_train, y_train_pred_after)
acc_test_after = accuracy_score(y_test, y_test_pred_after)

In [14]:
depth_after = get_tree_depth(tree.root)
leaves_after = get_leaf_count(tree.root)
acc_test_after = accuracy_score(y_test, tree.predict(X_test))

In [36]:
from sklearn.metrics import classification_report

print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ СРАВНЕНИЯ КАЧЕСТВА КЛАССИФИКАЦИИ")
print("="*50)
print("До редукции (Переобученное дерево):")
print(f"  - Точность на Train : {acc_train_before:.3%}")
print(f"  - Точность на Test  : {acc_test_before:.3%}")
print("-"*50)
print("После редукции (Оптимизированное дерево):")
print(f"  - Точность на Train : {acc_train_after:.3%}")
print(f"  - Точность на Test  : {acc_test_after:.3%}")
print("="*50)

print("\nДетальный отчет для тестовой выборки ПОСЛЕ редукции:")
print(classification_report(y_test, y_test_pred_after, target_names=['Погибли', 'Выжили'], digits=4))
print(classification_report(y_test, y_test_pred_before, target_names=['Погибли', 'Выжили'], digits=4))


РЕЗУЛЬТАТЫ СРАВНЕНИЯ КАЧЕСТВА КЛАССИФИКАЦИИ
До редукции (Переобученное дерево):
  - Точность на Train : 98.127%
  - Точность на Test  : 82.682%
--------------------------------------------------
После редукции (Оптимизированное дерево):
  - Точность на Train : 88.764%
  - Точность на Test  : 81.006%

Детальный отчет для тестовой выборки ПОСЛЕ редукции:
              precision    recall  f1-score   support

     Погибли     0.8447    0.8286    0.8365       105
      Выжили     0.7632    0.7838    0.7733        74

    accuracy                         0.8101       179
   macro avg     0.8039    0.8062    0.8049       179
weighted avg     0.8110    0.8101    0.8104       179

              precision    recall  f1-score   support

     Погибли     0.8627    0.8381    0.8502       105
      Выжили     0.7792    0.8108    0.7947        74

    accuracy                         0.8268       179
   macro avg     0.8210    0.8245    0.8225       179
weighted avg     0.8282    0.8268    0.8273  

In [ ]:
print("\n" + "="*60)
print(" СРАВНЕНИЕ СТРУКТУРЫ ДЕРЕВА ДО И ПОСЛЕ РЕДУКЦИИ (ПРУНИНГА)")
print("="*60)

# Функция для отрисовки мини-гистограммы в консоли
def draw_bar(val_before, val_after, label):
    max_val = max(val_before, val_after, 1)
    scale = 30 / max_val # масштабируем до 30 символов
    bar_before = "█" * int(val_before * scale)
    bar_after  = "█" * int(val_after * scale)
    print(f"{label}:")
    print(f"  До прунинга: [{val_before:<4}] {bar_before}")
    print(f"  После прунинга: [{val_after:<4}] {bar_after}")
    print("-" * 60)

draw_bar(depth_before, depth_after, "Максимальная глубина дерева")
draw_bar(leaves_before, leaves_after, "Общее количество листьев")

# Итоговый вывод эффективности
saved_leaves = leaves_before - leaves_after
print("Результат оптимизации:")
print(f"  • Удалено лишних листьев (ошибочных ветвей): {saved_leaves} (снижение на {saved_leaves/leaves_before:.1%})")
print(f"  • Изменение точности на тест-выборке: {acc_test_before:.2%} -> {acc_test_after:.2%}")
print("="*60)


 СРАВНЕНИЕ СТРУКТУРЫ ДЕРЕВА ДО И ПОСЛЕ РЕДУКЦИИ (ПРУНИНГА)
Максимальная глубина дерева:
  До прунинга:    [23  ] ██████████████████████████████
  После прунинга: [19  ] ████████████████████████
------------------------------------------------------------
Общее количество листьев:
  До прунинга:    [254 ] ██████████████████████████████
  После прунинга: [61  ] ███████
------------------------------------------------------------
Результат оптимизации:
  • Удалено лишних листьев (ошибочных ветвей): 193 (снижение на 76.0%)
  • Изменение точности на тест-выборке: 82.68% -> 81.01%


In [29]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

X_train_full = np.vstack([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

print("Обучение эталонного дерева решений sklearn...")
sklearn_tree = DecisionTreeClassifier(random_state=42, max_depth=10, criterion='gini')
sklearn_tree.fit(X_train_full, y_train_full)

Обучение эталонного дерева решений sklearn...


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",10
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",42
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current n

In [30]:
y_test_pred_sk = sklearn_tree.predict(X_test)

In [31]:
from sklearn.metrics import precision_score, recall_score, f1_score

acc_sk = accuracy_score(y_test, y_test_pred_sk)
prec_sk = precision_score(y_test, y_test_pred_sk)
rec_sk = recall_score(y_test, y_test_pred_sk)
f1_sk = f1_score(y_test, y_test_pred_sk)

In [32]:
print("\n" + "="*50)
print("МЕТРИКИ ЭТАЛОННОГО АЛГОРИТМА (SKLEARN)")
print("="*50)
print(f"Accuracy  (Общая точность)    : {acc_sk:.2%}")
print(f"Precision (Точность выживших) : {prec_sk:.2%}")
print(f"Recall    (Полнота выживших)  : {rec_sk:.2%}")
print(f"F1-Score  (Баланс метрик)     : {f1_sk:.2%}")
print("="*50)

print("\nПолный отчет классификации эталона:")
print(classification_report(y_test, y_test_pred_sk, target_names=['Погибли', 'Выжили'], digits=4))


МЕТРИКИ ЭТАЛОННОГО АЛГОРИТМА (SKLEARN)
Accuracy  (Общая точность)    : 81.01%
Precision (Точность выживших) : 81.25%
Recall    (Полнота выживших)  : 70.27%
F1-Score  (Баланс метрик)     : 75.36%

Полный отчет классификации эталона:
              precision    recall  f1-score   support

     Погибли     0.8087    0.8857    0.8455       105
      Выжили     0.8125    0.7027    0.7536        74

    accuracy                         0.8101       179
   macro avg     0.8106    0.7942    0.7995       179
weighted avg     0.8103    0.8101    0.8075       179



In [35]:
sklearn_tree.get_depth()

10